In [22]:
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors
import ast
import re



In [23]:

#cambiar como sea relevante la direccion. esto asume esta en el mismo directorio
df = pd.read_csv("./libros.csv")

print(f"Cantidad de libros: {len(df)}")
df.head()

Cantidad de libros: 120


,id,titulo,autores,generos,serie,sinopsis,url_libro,categoria_origen,fecha_extraccion
0,1,El papá de las bellezas,"[""Felipe Trigo""]","[""Realista"", ""Relato""]",NaN,"El papá de las bellezas , una obra maestra de ...",https://ww3.lectulandia.com/book/el-papa-de-la...,Realismo,2026-09-11
1,2,La última cerilla,"[""Marie Vareille""]","[""Novela"", ""Psicológico"", ""Realista""]",NaN,Las cifras no mienten: tres de cada cuatro niñ...,https://ww3.lectulandia.com/book/la-ultima-cer...,Realismo,2026-09-11
2,3,La enjuta,"[""Víctor Català""]","[""Realista"", ""Relato""]",NaN,En estos relatos la autora presenta una radiog...,https://ww3.lectulandia.com/book/la-enjuta/,Realismo,2026-09-11
3,4,En el camping,"[""Soledad Puértolas""]","[""Realista"", ""Relato""]",NaN,El esplendor de Soledad Puértolas en su madure...,https://ww3.lectulandia.com/book/en-el-camping/,Realismo,2026-09-11
4,5,Koljós,"[""Emmanuel Carrère""]","[""Ensayo"", ""Memorias"", ""Novela"", ""Realista""]",NaN,"Un libro total, una construcción narrativa des...",https://ww3.lectulandia.com/book/koljos/,Realismo,2026-09-11


In [24]:
def convertir_a_string(valor):
    if pd.isna(valor):
        return ""
    
    if isinstance(valor, str):
        try:
            valor_parseado = ast.literal_eval(valor)
            
            if isinstance(valor_parseado, list):
                return " ".join(str(x) for x in valor_parseado)
            
            return str(valor_parseado)
        
        except (ValueError, SyntaxError):
            return valor
    
    if isinstance(valor, list):
        return " ".join(str(x) for x in valor)
    
    return str(valor)


In [25]:
columnas = [
    "titulo",
    "sinopsis"
]

for columna in columnas:
    df[columna] = df[columna].apply(convertir_a_string)


CAMBIAR LA VARIABLE DE PURO!!


In [26]:
df["puro"] = df[columnas].agg(" ".join, axis=1)

df["puro"] = (
    df["puro"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [27]:
df[["titulo", "autores", "generos", "serie", "sinopsis", "puro"]].head()


,titulo,autores,generos,serie,sinopsis,puro
0,El papá de las bellezas,"[""Felipe Trigo""]","[""Realista"", ""Relato""]",NaN,"El papá de las bellezas , una obra maestra de ...",El papá de las bellezas El papá de las belleza...
1,La última cerilla,"[""Marie Vareille""]","[""Novela"", ""Psicológico"", ""Realista""]",NaN,Las cifras no mienten: tres de cada cuatro niñ...,La última cerilla Las cifras no mienten: tres ...
2,La enjuta,"[""Víctor Català""]","[""Realista"", ""Relato""]",NaN,En estos relatos la autora presenta una radiog...,La enjuta En estos relatos la autora presenta ...
3,En el camping,"[""Soledad Puértolas""]","[""Realista"", ""Relato""]",NaN,El esplendor de Soledad Puértolas en su madure...,En el camping El esplendor de Soledad Puértola...
4,Koljós,"[""Emmanuel Carrère""]","[""Ensayo"", ""Memorias"", ""Novela"", ""Realista""]",NaN,"Un libro total, una construcción narrativa des...","Koljós Un libro total, una construcción narrat..."


limpieza!!

In [28]:
import re

STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como",
    "con", "contra", "cual", "cuando", "de", "del", "desde", "donde",
    "dos", "el", "ella", "ellas", "ellos", "en", "entre", "era", "es",
    "esa", "esas", "ese", "eso", "esos", "esta", "estas", "este", "esto",
    "estos", "fue", "ha", "han", "hasta", "hay", "la", "las", "le", "les",
    "lo", "los", "más", "me", "mi", "mis", "muy", "no", "nos", "o", "para",
    "pero", "por", "que", "qué", "se", "sea", "ser", "si", "sí", "sin",
    "sobre", "son", "su", "sus", "también", "te", "tiene", "tienen", "todo",
    "todos", "tu", "tus", "un", "una", "uno", "unos", "y", "ya"
}



In [29]:
def limpiar_texto(texto):
    # 1. Lowercase
    texto = texto.lower()

    # 2. Eliminar signos de puntuación
    texto = re.sub(r"[^\w\s]", "", texto)

    # 3. Normalizar espacios
    texto = re.sub(r"\s+", " ", texto).strip()

    
    # 5. Remover stopwords
    palabras = texto.split()
    palabras = [
        palabra for palabra in palabras
        if palabra not in STOPWORDS_ES
    ]

    return " ".join(palabras)


# Aplicar a la columna "puro"
df["puro"] = df["puro"].apply(limpiar_texto)

In [30]:
df["puro"].head()

0    papá bellezas papá bellezas obra maestra felip...
1    última cerilla cifras mienten tres cada cuatro...
2    enjuta relatos autora presenta radiografía des...
3    camping esplendor soledad puértolas madurez di...
4    koljós libro total construcción narrativa desl...
Name: puro, dtype: str

# Parte B


## Prearmado

In [31]:

# 1. Cargar el modelo preentrenado
modelo = KeyedVectors.load_word2vec_format(
    "SBW-vectors-300-min5.bin",
    binary=True
)


In [32]:


# 2. Función para vectorizar cada libro
def vectorizar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        return np.zeros(300, dtype=np.float32)

    palabras = texto.split()

    # Conservar únicamente las palabras conocidas por el modelo
    vectores = [
        modelo[palabra]
        for palabra in palabras
        if palabra in modelo
    ]

    # Si ninguna palabra existe en el vocabulario
    if not vectores:
        return np.zeros(300, dtype=np.float32)

    # Promediar los vectores de las palabras
    return np.mean(vectores, axis=0)


In [33]:

# Crear un DataFrame nuevo con las columnas id y vec_sbw
df_sbw = df[["id", "puro"]].copy()

df_sbw["vec_sbw"] = df_sbw["puro"].apply(vectorizar_texto)

# Conservar únicamente las columnas necesarias
df_sbw = df_sbw[["id", "vec_sbw"]]

# Verificar dimensiones del vector
print(df_sbw["vec_sbw"].iloc[0].shape)  # (300,)

# Mostrar las primeras filas
df_sbw.head()

(300,)


,id,vec_sbw
0,1,"[0.01668873, -0.11789749, 0.017378552, -0.1071..."
1,2,"[-0.039745085, -0.06797318, 0.075977966, -0.09..."
2,3,"[-0.0020147818, -0.11794954, 0.075477146, -0.1..."
3,4,"[0.010127935, -0.0721531, 0.057110112, -0.0673..."
4,5,"[-0.010108512, -0.07284986, 0.024867795, -0.11..."


## Armado de cero

In [34]:
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec

In [35]:
textos = df["puro"].astype(str)

# Separar en oraciones y tokenizar
oraciones = []

for texto in textos:
    frases = re.split(r'[.!?¿¡\n]+', texto)

    for frase in frases:
        palabras = simple_preprocess(
            frase,
            deacc=True,  # Quita acentos
            min_len=2
        )

        if palabras:
            oraciones.append(palabras)

cambiamos workers a 1 y seed a 42 por reproducibilidad

In [ ]:

# PRUEBA de 100 5 2 10
modelo_propio = Word2Vec(
    sentences=oraciones,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    seed=42,
    epochs=10
)

print("Cantidad de oraciones:", len(oraciones))
print("Tamaño del vocabulario:", len(modelo_propio.wv))

Cantidad de oraciones: 120
Tamaño del vocabulario: 1677


In [67]:
# Testing
palabras = ["perro", "gato", "casa", "computadora"]

for palabra in palabras:
    print(palabra, palabra in modelo_propio.wv)
    
try:
    similares = modelo_propio.wv.most_similar("casa")
    print(similares)

except KeyError:
    print('La palabra no está en el vocabulario del modelo.')


perro True
gato False
casa True
computadora False
[('marie', 0.9975618720054626), ('raices', 0.9974915385246277), ('zoe', 0.9974802732467651), ('falsas', 0.9974244236946106), ('magnitud', 0.9973868131637573), ('casarse', 0.9973751902580261), ('narraciones', 0.9973682165145874), ('rica', 0.9973652958869934), ('sofia', 0.9973404407501221), ('famoso', 0.9973319172859192)]


In [68]:

# PRUEBA con 150 5 5 20
modelo_propio = Word2Vec(
    sentences=oraciones,
    vector_size=150,
    window=3,
    min_count=3,
    sg=1,
    workers=1,
    seed=42,
    epochs=20
)

print("Cantidad de oraciones:", len(oraciones))
print("Tamaño del vocabulario:", len(modelo_propio.wv))

Cantidad de oraciones: 120
Tamaño del vocabulario: 838


In [70]:
# Testing
palabras = ["perro", "gato", "casa", "computadora"]

for palabra in palabras:
    print(palabra, palabra in modelo_propio.wv)
    
try:
    similares = modelo_propio.wv.most_similar("casa")
    print(similares)

except KeyError:
    print('La palabra no está en el vocabulario del modelo.')

perro True
gato False
casa True
computadora False
[('marie', 0.9975618720054626), ('raices', 0.9974915385246277), ('zoe', 0.9974802732467651), ('falsas', 0.9974244236946106), ('magnitud', 0.9973868131637573), ('casarse', 0.9973751902580261), ('narraciones', 0.9973682165145874), ('rica', 0.9973652958869934), ('sofia', 0.9973404407501221), ('famoso', 0.9973319172859192)]
